In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling import CATEGORICAL_FEATURES

previous_cache = project_root / "data/processed/lab05"
frames = {}

for split in ["train", "validation"]:
    frame = pd.read_parquet(
        project_root / f"data/processed/lab02_{split}.parquet",
        columns=["source_row", *CATEGORICAL_FEATURES],
    )

    np.testing.assert_array_equal(
        frame["source_row"].to_numpy(),
        np.load(previous_cache / f"{split}_source_rows.npy"),
    )

    frames[split] = frame

In [2]:
encoder = OrdinalEncoder(
    dtype=np.int64,
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-1,
)

category_ids = {
    "train": (
        encoder.fit_transform(frames["train"][CATEGORICAL_FEATURES])
        + 1
    ),
    "validation": (
        encoder.transform(frames["validation"][CATEGORICAL_FEATURES])
        + 1
    ),
}

vocabulary_sizes = [
    len(categories) + 1
    for categories in encoder.categories_
]

print(dict(zip(CATEGORICAL_FEATURES, vocabulary_sizes)))

{'payment_currency': 16, 'receiving_currency': 16, 'payment_format': 7}


In [11]:
payment_format_mapping = {
    category: index + 1
    for index, category in enumerate(encoder.categories_[2])
}

payment_format_mapping

{'ACH': 1, 'Bitcoin': 2, 'Cash': 3, 'Cheque': 4, 'Credit Card': 5, 'Wire': 6}

In [15]:
probe = (
    frames["validation"][CATEGORICAL_FEATURES]
    .head(1)
    .astype("string")
    .copy()
)

probe.loc[:, "payment_format"] = "NEW_PAYMENT_RAIL"

probe_ids = encoder.transform(probe) + 1

assert probe_ids[0, 2] == 0
print("Unseen payment format maps to ID 0.")

Unseen payment format maps to ID 0.


In [22]:
cache_dir = project_root / "data/processed/lab06"
cache_dir.mkdir(parents=True, exist_ok=True)

for split, ids in category_ids.items():
    assert ids.shape == (len(frames[split]), 3)
    assert ids.dtype == np.int64

    for column_index, size in enumerate(vocabulary_sizes):
        assert (ids[:, column_index] >= 0).all()
        assert (ids[:, column_index] < size).all()

    np.save(cache_dir / f"X_cat_{split}.npy", ids)

    print(
        split,
        ids.shape,
        ids.dtype,
        f"{ids.nbytes / 1024**2:.1f} MiB",
    )

joblib.dump(encoder, cache_dir / "category_encoder.joblib")

train (1379551, 3) int64 31.6 MiB
validation (965524, 3) int64 22.1 MiB


['/Users/sara/Projects/data-science-lab/data/processed/lab06/category_encoder.joblib']

In [1]:
import sys
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.embeddings import EmbeddingClassifier, predict_embedding_scores

torch.set_num_threads(2)

previous_cache = project_root / "data/processed/lab05"
cache_dir = project_root / "data/processed/lab06"

preprocessor = joblib.load(previous_cache / "preprocessor.joblib")
encoder = joblib.load(cache_dir / "category_encoder.joblib")

# Lab 5 placed its numerical features before its one-hot columns.
n_numeric = preprocessor.output_indices_["categorical"].start
assert n_numeric == 8

vocabulary_sizes = tuple(len(c) + 1 for c in encoder.categories_)
embedding_dims = (4, 4, 3)

In [2]:
def load_inputs(split):
    previous_inputs = np.load(
        previous_cache / f"X_{split}.npy",
        mmap_mode="r",
    )

    # Copy only the eight numerical columns into a writable array.
    numeric = np.array(
        previous_inputs[:, :n_numeric],
        dtype=np.float32,
        order="C",
        copy=True,
    )
    categories = np.load(cache_dir / f"X_cat_{split}.npy")
    labels = np.load(previous_cache / f"y_{split}.npy")

    assert numeric.shape == (len(labels), n_numeric)
    assert categories.shape == (len(labels), len(vocabulary_sizes))
    assert categories.dtype == np.int64

    return (
        torch.from_numpy(numeric),
        torch.from_numpy(categories),
        torch.from_numpy(labels),
    )


train_numeric, train_categories, y_train = load_inputs("train")
val_numeric, val_categories, y_validation = load_inputs("validation")

for name, numeric, categories in [
    ("train", train_numeric, train_categories),
    ("validation", val_numeric, val_categories),
]:
    size_bytes = sum(
        tensor.numel() * tensor.element_size()
        for tensor in [numeric, categories]
    )
    print(name, f"{size_bytes / 1024**2:.1f} MiB")

train 73.7 MiB
validation 51.6 MiB


In [3]:
torch.manual_seed(42)

positive_rate = y_train.mean().item()

model = EmbeddingClassifier(
    n_numeric=n_numeric,
    vocabulary_sizes=vocabulary_sizes,
    embedding_dims=embedding_dims,
    positive_rate=positive_rate,
)

# Keep a snapshot so we can inspect what changed in Step 3.
initial_payment_embeddings = (
    model.embeddings[2].weight.detach().clone()
)

train_loader = DataLoader(
    TensorDataset(train_numeric, train_categories, y_train),
    batch_size=4096,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

epochs = 5
training_history = []

for epoch in range(1, epochs + 1):
    start = perf_counter()
    model.train()
    total_loss = 0.0

    for numeric_batch, category_batch, labels in train_loader:
        optimizer.zero_grad(set_to_none=True)

        logits = model(numeric_batch, category_batch)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)

    validation_scores = predict_embedding_scores(
        model,
        val_numeric,
        val_categories,
    )
    validation_ap = average_precision_score(
        y_validation.numpy(),
        validation_scores,
    )

    result = {
        "epoch": epoch,
        "train_loss": total_loss / len(y_train),
        "validation_ap": validation_ap,
        "seconds": perf_counter() - start,
    }
    training_history.append(result)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"loss={result['train_loss']:.5f} | "
        f"validation AP={validation_ap:.4f} | "
        f"{result['seconds']:.1f}s",
        flush=True,
    )

history = pd.DataFrame(training_history)
history

Epoch 1/5 | loss=0.00807 | validation AP=0.0296 | 7.2s
Epoch 2/5 | loss=0.00705 | validation AP=0.0405 | 6.3s
Epoch 3/5 | loss=0.00689 | validation AP=0.0465 | 5.9s
Epoch 4/5 | loss=0.00678 | validation AP=0.0606 | 6.9s
Epoch 5/5 | loss=0.00669 | validation AP=0.0767 | 6.0s


,epoch,train_loss,validation_ap,seconds
0,1,0.008068,0.029594,7.159236
1,2,0.007050,0.040482,6.348567
2,3,0.006889,0.046484,5.882519
3,4,0.006783,0.060550,6.876739
4,5,0.006692,0.076726,6.032978


In [4]:
reports_dir = project_root / "reports"
models_dir = project_root / "models"
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

history.to_csv(
    reports_dir / "lab_06_training.csv",
    index=False,
)

np.save(cache_dir / "validation_scores.npy", validation_scores)
np.save(
    cache_dir / "payment_embeddings_initial.npy",
    initial_payment_embeddings.numpy(),
)

torch.save(
    {
        "model_state": model.state_dict(),
        "n_numeric": n_numeric,
        "vocabulary_sizes": vocabulary_sizes,
        "embedding_dims": embedding_dims,
        "positive_rate": positive_rate,
        "epochs": epochs,
        "categorical_columns": encoder.feature_names_in_.tolist(),
    },
    models_dir / "lab06_embedding_classifier.pt",
)